# 03 · Modelado y Evaluación — LoanSight

Entrenamiento y comparación de modelos para las dos tareas:
- **Regresión**: `int_rate` (LinearRegression vs RandomForest).
- **Clasificación**: `is_default` (LogReg, KNN, DecisionTree, RandomForest).

`random_state=42`, split estratificado en clasificación, métricas apropiadas al desbalanceo. El código replica el de `backend/app/models/train.py`.

In [3]:
import sys
from pathlib import Path

import duckdb
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sys.path.insert(0, str(Path.cwd().parent / "backend"))

DB_PATH = Path.cwd().parent / "data" / "processed" / "loansight.duckdb"
con = duckdb.connect(str(DB_PATH), read_only=True)
print("Warehouse:", DB_PATH.exists())


ModuleNotFoundError: No module named 'matplotlib'

In [ ]:
df = con.execute('''
    SELECT f.loan_amnt, f.term_months, f.annual_inc, f.dti,
           m.emp_length_years, p.purpose, g.grade,
           f.int_rate, f.loan_status, f.is_default
    FROM FACT_LOANS f
    JOIN DIM_PROPOSITO p ON f.proposito_sk = p.proposito_sk
    JOIN DIM_EMPLEO    m ON f.empleo_sk    = m.empleo_sk
    JOIN DIM_GRADO     g ON f.grado_sk     = g.grado_sk
''').df()
print(df.shape)
df.head()


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (root_mean_squared_error, r2_score, mean_absolute_error,
    roc_auc_score, f1_score, precision_score, recall_score, accuracy_score,
    confusion_matrix, roc_curve)
from app.preprocessing.pipeline import build_model_pipeline, MODEL_COLUMNS
SEED = 42

## 1. Regresión — predicción de `int_rate`

In [ ]:
reg = df.dropna(subset=['int_rate']).sample(80_000, random_state=SEED)
Xr, yr = reg[MODEL_COLUMNS], reg['int_rate']
Xtr, Xte, ytr, yte = train_test_split(Xr, yr, test_size=0.2, random_state=SEED)

reg_models = {
    'LinearRegression': LinearRegression(),
    'RandomForestRegressor': RandomForestRegressor(
        n_estimators=100, max_depth=12, min_samples_leaf=50, n_jobs=-1, random_state=SEED)}
rows = []
for name, est in reg_models.items():
    pipe = build_model_pipeline(est); pipe.fit(Xtr, ytr)
    pred = pipe.predict(Xte)
    rows.append({'modelo':name,
        'RMSE':root_mean_squared_error(yte,pred),
        'MAE':mean_absolute_error(yte,pred),
        'R2':r2_score(yte,pred)})
pd.DataFrame(rows).round(3)

## 2. Clasificación — predicción de default
Split **estratificado** por el desbalanceo; `class_weight='balanced'` donde aplica.

In [ ]:
clf = df.dropna(subset=['is_default']).copy()
clf['is_default'] = clf['is_default'].astype(int)
clf = clf.sample(80_000, random_state=SEED)
Xc, yc = clf[MODEL_COLUMNS], clf['is_default']
Xtr, Xte, ytr, yte = train_test_split(Xc, yc, test_size=0.2, random_state=SEED, stratify=yc)

clf_models = {
    'LogisticRegression': LogisticRegression(max_iter=1000, class_weight='balanced', random_state=SEED),
    'KNeighborsClassifier': KNeighborsClassifier(n_neighbors=25, n_jobs=-1),
    'DecisionTreeClassifier': DecisionTreeClassifier(max_depth=8, class_weight='balanced', random_state=SEED),
    'RandomForestClassifier': RandomForestClassifier(n_estimators=120, max_depth=14,
        min_samples_leaf=20, class_weight='balanced', n_jobs=-1, random_state=SEED)}
fitted, rows = {}, []
for name, est in clf_models.items():
    pipe = build_model_pipeline(est); pipe.fit(Xtr, ytr); fitted[name] = pipe
    pred = pipe.predict(Xte); proba = pipe.predict_proba(Xte)[:,1]
    rows.append({'modelo':name, 'AUC':roc_auc_score(yte,proba),
        'F1':f1_score(yte,pred), 'Precision':precision_score(yte,pred,zero_division=0),
        'Recall':recall_score(yte,pred), 'Accuracy':accuracy_score(yte,pred)})
results = pd.DataFrame(rows).round(3).sort_values('AUC', ascending=False)
results

## 3. Curva ROC y matriz de confusión del mejor clasificador

In [ ]:
best = results.iloc[0]['modelo']
pipe = fitted[best]
proba = pipe.predict_proba(Xte)[:,1]; pred = pipe.predict(Xte)
fpr, tpr, _ = roc_curve(yte, proba)

fig, axes = plt.subplots(1,2, figsize=(12,4.5))
axes[0].plot(fpr, tpr, color='#3fb950', lw=2, label=f'{best} (AUC={roc_auc_score(yte,proba):.3f})')
axes[0].plot([0,1],[0,1],'--', color='gray')
axes[0].set_xlabel('FPR'); axes[0].set_ylabel('TPR'); axes[0].set_title('Curva ROC'); axes[0].legend()

cm = confusion_matrix(yte, pred)
im = axes[1].imshow(cm, cmap='Blues')
for (i,j), v in np.ndenumerate(cm):
    axes[1].text(j, i, f'{v:,}', ha='center', va='center')
axes[1].set_xticks([0,1]); axes[1].set_xticklabels(['Pagado','Default'])
axes[1].set_yticks([0,1]); axes[1].set_yticklabels(['Pagado','Default'])
axes[1].set_xlabel('Predicho'); axes[1].set_ylabel('Real'); axes[1].set_title(f'Matriz de confusión — {best}')
plt.tight_layout(); plt.show()

NameError: name 'results' is not defined

## Conclusiones del modelado
- **Regresión**: RandomForest supera a LinearRegression en RMSE/R², aunque el techo es bajo al excluir `grade` (no disponible en la API).
- **Clasificación**: RandomForest y LogisticRegression lideran en AUC; KNN tiene recall muy bajo (mala opción con este desbalanceo).
- La **accuracy engaña** (un clasificador trivial 'todo pagado' daría ~80%); por eso priorizamos AUC-ROC y F1.
- Los mejores modelos se serializan con `train.py` a `backend/app/models/artifacts/` e impulsan la API y el frontend.